In [3]:
import os
import time
import json
import math
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from web3 import Web3


# ============================================================
# 1. Load .env
# ============================================================

load_dotenv()

PRIVATE_KEY = os.getenv("PRIVATE_KEY")

if not PRIVATE_KEY:
    raise ValueError("PRIVATE_KEY is missing in .env")

if PRIVATE_KEY.startswith("0x"):
    PRIVATE_KEY = PRIVATE_KEY[2:]


# ============================================================
# 2. Minimal ABI for MileageLedger
# ============================================================

MILEAGE_LEDGER_ABI = [
    {
        "inputs": [
            {"internalType": "string", "name": "vehicleId", "type": "string"},
            {"internalType": "string", "name": "timestamp", "type": "string"},
            {"internalType": "string", "name": "proofCid", "type": "string"},
            {"internalType": "uint256", "name": "odometerKm", "type": "uint256"},
        ],
        "name": "recordMileage",
        "outputs": [],
        "stateMutability": "nonpayable",
        "type": "function",
    },
    {
        "inputs": [
            {"internalType": "string", "name": "vehicleId", "type": "string"}
        ],
        "name": "getLastOdometerKm",
        "outputs": [
            {"internalType": "uint256", "name": "", "type": "uint256"}
        ],
        "stateMutability": "view",
        "type": "function",
    },
    {
        "inputs": [
            {"internalType": "string", "name": "vehicleId", "type": "string"}
        ],
        "name": "getFraudFlag",
        "outputs": [
            {"internalType": "bool", "name": "", "type": "bool"}
        ],
        "stateMutability": "view",
        "type": "function",
    },
]


# ============================================================
# 3. Networks
# ============================================================

NETWORKS = [
    {
        "name": "iota_testnet",
        "label": "IOTA EVM Testnet",
        "rpc_url": os.getenv("IOTA_RPC_URL"),
        "contract_address": os.getenv("IOTA_CONTRACT_ADDRESS"),
        "native_symbol": "IOTA",
        "usd_price": float(os.getenv("IOTA_USD", "0.08570490")),
    },
    {
        "name": "sepolia",
        "label": "Ethereum Sepolia",
        "rpc_url": os.getenv("SEPOLIA_RPC_URL"),
        "contract_address": os.getenv("SEPOLIA_CONTRACT_ADDRESS"),
        "native_symbol": "ETH",
        "usd_price": float(os.getenv("ETH_USD", "3000")),
    },
    #{
    #   "name": "amoy",
    #    "label": "Polygon Amoy",
    #    "rpc_url": os.getenv("AMOY_RPC_URL"),
    #    "contract_address": os.getenv("AMOY_CONTRACT_ADDRESS"),
    #    "native_symbol": "POL",
    #    "usd_price": float(os.getenv("POL_USD", "0.5")),
    #},
]


# ============================================================
# 4. Benchmark parameters
# ============================================================

NORMAL_TX_COUNT = 30
ROLLBACK_TX_COUNT = 10
POST_FRAUD_TX_COUNT = 10

BASE_ODOMETER = 1000
STEP = 10

SLEEP_BETWEEN_TX_SECONDS = 1.2
TX_TIMEOUT_SECONDS = 360

RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)

RAW_CSV_PATH = RESULTS_DIR / "benchmark_raw.csv"
SUMMARY_CSV_PATH = RESULTS_DIR / "benchmark_summary.csv"
SUMMARY_JSON_PATH = RESULTS_DIR / "benchmark_summary.json"


# ============================================================
# 5. Helpers
# ============================================================

def now_iso():
    return datetime.now(timezone.utc).isoformat()


def clean_address(address):
    if not address:
        return None
    return address.strip().replace('"', "").replace("'", "")


def make_fake_cid(network_name, scenario, index):
    return f"ipfs://{network_name}_{scenario}_{index}_encrypted_evidence_cid"


def connect_web3(rpc_url):
    w3 = Web3(Web3.HTTPProvider(rpc_url, request_kwargs={"timeout": 60}))

    if not w3.is_connected():
        raise ConnectionError(f"Cannot connect to RPC: {rpc_url}")

    return w3


def get_scenario_plan(network_name):
    unique_suffix = int(time.time())
    vehicle_id = f"V-{network_name.upper()}-{unique_suffix}"

    plan = []

    # Phase 1: 30 normal monotonic updates
    for i in range(NORMAL_TX_COUNT):
        plan.append({
            "scenario": "normal_update",
            "submitted_km": BASE_ODOMETER + i * STEP,
            "vehicle_id": vehicle_id,
        })

    last_valid_km = BASE_ODOMETER + (NORMAL_TX_COUNT - 1) * STEP

    # Phase 2: 10 rollback attempts
    for i in range(ROLLBACK_TX_COUNT):
        plan.append({
            "scenario": "rollback_attempt",
            "submitted_km": BASE_ODOMETER - 100 - i * STEP,
            "vehicle_id": vehicle_id,
        })

    # Phase 3: 10 valid updates after fraud
    for i in range(POST_FRAUD_TX_COUNT):
        plan.append({
            "scenario": "post_fraud_valid_update",
            "submitted_km": last_valid_km + (i + 1) * STEP,
            "vehicle_id": vehicle_id,
        })

    return plan


def safe_float(value):
    try:
        if value is None:
            return None
        return float(value)
    except Exception:
        return None


def percentile(values, p):
    values = sorted([v for v in values if v is not None and not math.isnan(v)])
    if not values:
        return None
    index = math.ceil((p / 100) * len(values)) - 1
    index = max(0, min(index, len(values) - 1))
    return values[index]


def check_contract_code(w3, contract_address):
    code = w3.eth.get_code(contract_address)
    if code in [b"", "0x"]:
        raise ValueError(f"No contract code found at {contract_address}")


def get_effective_gas_price(w3, tx_hash, receipt):
    if "effectiveGasPrice" in receipt and receipt["effectiveGasPrice"]:
        return int(receipt["effectiveGasPrice"])

    tx = w3.eth.get_transaction(tx_hash)
    if "gasPrice" in tx and tx["gasPrice"]:
        return int(tx["gasPrice"])

    return int(w3.eth.gas_price)



def build_transaction_params(w3, account_address, nonce, network_name):
    latest_block = w3.eth.get_block("latest")
    base_fee = latest_block.get("baseFeePerGas", None)

    # Sepolia / Ethereum-like networks: use EIP-1559
    if base_fee is not None and network_name in ["sepolia", "amoy"]:
        priority_fee = w3.to_wei(2, "gwei")

        # multiplier pour éviter underpriced tx
        max_fee = int(base_fee * 2 + priority_fee)

        return {
            "from": account_address,
            "nonce": nonce,
            "maxFeePerGas": max_fee,
            "maxPriorityFeePerGas": priority_fee,
        }

    # IOTA EVM or legacy-style RPC
    gas_price = int(w3.eth.gas_price * 1.20)

    return {
        "from": account_address,
        "nonce": nonce,
        "gasPrice": gas_price,
    }


def estimate_gas_safely(function_call, tx_params):
    try:
        estimated = function_call.estimate_gas(tx_params)
        return int(estimated * 1.25)
    except Exception:
        return 500000


# ============================================================
# 6. Execute one transaction
# ============================================================

def execute_one_transaction(
    w3,
    contract,
    account,
    network,
    tx_index,
    plan_item,
    nonce,
):
    timestamp = now_iso()
    proof_cid = make_fake_cid(network["name"], plan_item["scenario"], tx_index)

    row = {
        "network": network["name"],
        "network_label": network["label"],
        "scenario": plan_item["scenario"],
        "tx_index": tx_index,
        "vehicle_id": plan_item["vehicle_id"],
        "timestamp": timestamp,
        "submitted_km": plan_item["submitted_km"],
        "proof_cid": proof_cid,
        "tx_hash": "",
        "block_number": "",
        "gas_used": "",
        "effective_gas_price_wei": "",
        "cost_native": "",
        "native_symbol": network["native_symbol"],
        "cost_usd": "",
        "latency_s": "",
        "success": False,
        "error_type": "",
        "error_message": "",
        "stored_km_after_tx": "",
        "fraud_flag_after_tx": "",
        "sender": account.address,
        "contract_address": network["contract_address"],
    }

    try:
        function_call = contract.functions.recordMileage(
            plan_item["vehicle_id"],
            timestamp,
            proof_cid,
            int(plan_item["submitted_km"]),
        )

        tx_params = build_transaction_params(
            w3=w3,
            account_address=account.address,
            nonce=nonce,
            network_name=network["name"],
        )

        tx_params["gas"] = estimate_gas_safely(function_call, tx_params)

        transaction = function_call.build_transaction(tx_params)

        signed_tx = account.sign_transaction(transaction)

        start_time = time.time()
        tx_hash = w3.eth.send_raw_transaction(signed_tx.rawTransaction)
        row["tx_hash"] = tx_hash.hex()

        receipt = w3.eth.wait_for_transaction_receipt(
            tx_hash,
            timeout=TX_TIMEOUT_SECONDS,
        )
        end_time = time.time()

        gas_used = int(receipt["gasUsed"])
        effective_gas_price = get_effective_gas_price(w3, tx_hash, receipt)

        cost_wei = gas_used * effective_gas_price
        cost_native = float(w3.from_wei(cost_wei, "ether"))
        cost_usd = cost_native * float(network["usd_price"])

        stored_km = contract.functions.getLastOdometerKm(
            plan_item["vehicle_id"]
        ).call()

        fraud_flag = contract.functions.getFraudFlag(
            plan_item["vehicle_id"]
        ).call()

        row["block_number"] = int(receipt["blockNumber"])
        row["gas_used"] = gas_used
        row["effective_gas_price_wei"] = effective_gas_price
        row["cost_native"] = cost_native
        row["cost_usd"] = cost_usd
        row["latency_s"] = round(end_time - start_time, 4)
        row["success"] = bool(receipt["status"] == 1)
        row["stored_km_after_tx"] = int(stored_km)
        row["fraud_flag_after_tx"] = int(bool(fraud_flag))

        return row, nonce + 1

    except Exception as e:
        row["success"] = False
        row["error_type"] = type(e).__name__
        row["error_message"] = str(e).replace("\n", " ")[:500]

        # Important: resynchroniser le nonce après une erreur
        try:
            new_nonce = w3.eth.get_transaction_count(account.address, "pending")
        except Exception:
            new_nonce = nonce + 1

        return row, new_nonce

# ============================================================
# 7. Summarize results
# ============================================================

def summarize_results(df):
    summary_rows = []

    for (network, scenario), group in df.groupby(["network", "scenario"]):
        success_group = group[group["success"] == True]

        gas_values = pd.to_numeric(success_group["gas_used"], errors="coerce").dropna().tolist()
        latency_values = pd.to_numeric(success_group["latency_s"], errors="coerce").dropna().tolist()
        cost_native_values = pd.to_numeric(success_group["cost_native"], errors="coerce").dropna().tolist()
        cost_usd_values = pd.to_numeric(success_group["cost_usd"], errors="coerce").dropna().tolist()

        first = group.iloc[0]

        summary_rows.append({
            "network": network,
            "network_label": first["network_label"],
            "scenario": scenario,
            "total_tx": len(group),
            "successful_tx": len(success_group),
            "failed_tx": len(group) - len(success_group),
            "success_rate_percent": round((len(success_group) / len(group)) * 100, 2),

            "gas_mean": round(pd.Series(gas_values).mean(), 2) if gas_values else None,
            "gas_median": round(pd.Series(gas_values).median(), 2) if gas_values else None,
            "gas_min": min(gas_values) if gas_values else None,
            "gas_max": max(gas_values) if gas_values else None,

            "latency_mean_s": round(pd.Series(latency_values).mean(), 4) if latency_values else None,
            "latency_median_s": round(pd.Series(latency_values).median(), 4) if latency_values else None,
            "latency_p95_s": round(percentile(latency_values, 95), 4) if latency_values else None,
            "latency_p99_s": round(percentile(latency_values, 99), 4) if latency_values else None,
            "latency_min_s": round(min(latency_values), 4) if latency_values else None,
            "latency_max_s": round(max(latency_values), 4) if latency_values else None,

            "cost_native_mean": round(pd.Series(cost_native_values).mean(), 12) if cost_native_values else None,
            "cost_usd_mean": round(pd.Series(cost_usd_values).mean(), 12) if cost_usd_values else None,
            "native_symbol": first["native_symbol"],
        })

    return pd.DataFrame(summary_rows)

def get_sleep_time(network_name):
    return 1.5

# ============================================================
# 8. Main benchmark
# ============================================================

def main():
    raw_rows = []

    active_networks = []

    for net in NETWORKS:
        rpc_url = net["rpc_url"]
        contract_address = clean_address(net["contract_address"])

        if not rpc_url or not contract_address:
            print(f"[SKIP] {net['name']} missing RPC URL or contract address.")
            continue

        net["contract_address"] = Web3.to_checksum_address(contract_address)
        active_networks.append(net)

    if not active_networks:
        raise ValueError("No active network found. Check your .env file.")

    for network in active_networks:
        print("\n============================================================")
        print(f"Starting benchmark on {network['label']}")
        print(f"Contract: {network['contract_address']}")
        print("============================================================")

        w3 = connect_web3(network["rpc_url"])
        account = w3.eth.account.from_key(PRIVATE_KEY)

        chain_id = w3.eth.chain_id
        print(f"Connected chain ID: {chain_id}")
        print(f"Wallet: {account.address}")

        balance = w3.eth.get_balance(account.address)
        print(f"Balance: {w3.from_wei(balance, 'ether')} {network['native_symbol']}")

        check_contract_code(w3, network["contract_address"])

        contract = w3.eth.contract(
            address=network["contract_address"],
            abi=MILEAGE_LEDGER_ABI,
        )

        nonce = w3.eth.get_transaction_count(account.address, "pending")
        plan = get_scenario_plan(network["name"])

        for i, item in enumerate(plan, start=1):
            nonce = w3.eth.get_transaction_count(account.address, "pending")

            print(
                f"[{network['name']}] TX {i}/{len(plan)} | "
                f"{item['scenario']} | km={item['submitted_km']} | nonce={nonce}"
            )

            row, nonce = execute_one_transaction(
                w3=w3,
                contract=contract,
                account=account,
                network=network,
                tx_index=i,
                plan_item=item,
                nonce=nonce,
            )

            raw_rows.append(row)

            if row["success"]:
                print(
                    f"  OK | gas={row['gas_used']} | "
                    f"latency={row['latency_s']}s | "
                    f"stored={row['stored_km_after_tx']} | "
                    f"fraud={row['fraud_flag_after_tx']} | "
                    f"cost={row['cost_native']} {row['native_symbol']}"
                )
            else:
                print(f"  FAILED | {row['error_type']} | {row['error_message']}")

            time.sleep(get_sleep_time(network["name"]))

    raw_df = pd.DataFrame(raw_rows)
    raw_df.to_csv(RAW_CSV_PATH, index=False)

    summary_df = summarize_results(raw_df)
    summary_df.to_csv(SUMMARY_CSV_PATH, index=False)

    with open(SUMMARY_JSON_PATH, "w", encoding="utf-8") as f:
        json.dump(summary_df.to_dict(orient="records"), f, indent=2)

    print("\n==================== SUMMARY ====================")
    print(summary_df.to_string(index=False))
    print("=================================================")

    print(f"\nRaw results saved to: {RAW_CSV_PATH}")
    print(f"Summary CSV saved to: {SUMMARY_CSV_PATH}")
    print(f"Summary JSON saved to: {SUMMARY_JSON_PATH}")


if __name__ == "__main__":
    main()

python-dotenv could not parse statement starting at line 2



Starting benchmark on IOTA EVM Testnet
Contract: 0xa03610f40588e9b07695A4c1478b629aa96C9E38
Connected chain ID: 1076
Wallet: 0xC40f4633b305b90cC0A048dfBF16BAf75Eb1b0C8
Balance: 14.7639146519 IOTA
[iota_testnet] TX 1/50 | normal_update | km=1000 | nonce=2320
  OK | gas=201974 | latency=2.0346s | stored=1000 | fraud=0 | cost=0.002423688 IOTA
[iota_testnet] TX 2/50 | normal_update | km=1010 | nonce=2321
  OK | gas=184896 | latency=1.7992s | stored=1010 | fraud=0 | cost=0.002218752 IOTA
[iota_testnet] TX 3/50 | normal_update | km=1020 | nonce=2322
  OK | gas=184896 | latency=1.7832s | stored=1020 | fraud=0 | cost=0.002218752 IOTA
[iota_testnet] TX 4/50 | normal_update | km=1030 | nonce=2323
  OK | gas=184896 | latency=1.6784s | stored=1030 | fraud=0 | cost=0.002218752 IOTA
[iota_testnet] TX 5/50 | normal_update | km=1040 | nonce=2324
  OK | gas=184896 | latency=1.7537s | stored=1040 | fraud=0 | cost=0.002218752 IOTA
[iota_testnet] TX 6/50 | normal_update | km=1050 | nonce=2325
  OK | gas=